# 01 — Sample Construction

Builds the fund-level (wficn × month) analysis panel used by every later notebook.

| | |
|---|---|
| **Inputs** | CRSP MFDB (WRDS `q_mutualfunds`): `monthly_tna_ret_nav.sas7bdat`, `fund_summary2.sas7bdat`; WRDS MFLINKS table `MFLINKS_1.txt` |
| **Output** | `analysis_panel.parquet` → read by NB03, NB06, NB08 |
| **Thesis** | Section 3.2, Table 1 (attrition waterfall), Table 3 (fund-level summary statistics) |
| **Expected result** | 2,138 funds, 373,387 fund-months, 2000-01 to 2022-06 (checked automatically in Section 6) |

**To replicate:** set `DATA_ROOT` in `thesis_config.py` (see README), then *Restart Kernel → Run All*.
The notebook is written to run top-to-bottom exactly once; no cell needs to be run in isolation.
The first run parses the SAS files and caches them as Parquet in `CACHE_DIR` (slow, ~minutes); later runs reuse the cache.
Peak memory is several GB because `fund_summary2` is wide; large intermediates are freed as soon as they are no longer needed.

**Pipeline** (step numbers match the screens listed in Section 3.2):

- Share-class level (`crsp_fundno`): point-in-time characteristics merge → (1) EQ asset code and U.S.-only geography → (2) index/ETF exclusion, plus variable-annuity and sales-restricted classes → (3) TNA ≥ \$50M → (4) drop missing returns → (5) NAV-consistency screen → (6) ≥ 60 consecutive months
- Link to `wficn` via MFLINKS, collapse share classes (lagged-TNA value weights), re-apply the 60-month screen
- Portfolio level (`wficn`): clean `per_cash`/`per_com` → (7) mandate screen → (8) equity-share screen → (9) pre-coverage index-month removal

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import pyreadstat

from thesis_config import (CRSP_DIR, DATA_DIR, MFLINKS_1, assert_matches, check_paths, max_consecutive_run,
                           read_sas_cached, save_parquet)

check_paths(CRSP_DIR, MFLINKS_1, DATA_DIR)
OUT_FILE = DATA_DIR / "analysis_panel.parquet"      # read by NB03 and NB08
print(f"pandas {pd.__version__} | numpy {np.__version__} | pyreadstat {pyreadstat.__version__}")

**Sample-definition parameters** (all judgment calls in one place).

In [ ]:
MIN_TNA        = 50      # $M, per fund-month (share-class level)
MIN_CONSEC     = 60      # consecutive monthly returns, share-class level and again at wficn level
PER_COM_FLOOR  = 80.0    # % common stock; fund-level time-series median (Cremers-Petajisto 2009)
PCT_BOUNDS     = (-100, 100)   # per_cash / per_com outside this range are set to NaN
INDEX_FLAG_COVERAGE_START = pd.Timestamp("2003-07-01")   # index_fund_flag coverage begins 2003-06-30 (CRSP guide)

# Step 1 — U.S.-only geography: share classes whose modal Lipper objective is international/global.
# (The factor model prices U.S. risk only; a foreign fund's cash counterfactual is not a U.S. index.)
INTERNATIONAL_OBJ = {
    "EM", "EMP",               # Emerging Markets
    "EU",                      # European Region
    "JA",                      # Japan
    "IF", "INR",               # International
    "XJ", "PC", "CH",          # Pacific ex-Japan, Pacific Region, China Region
    "LT",                      # Latin America
    "GL",                      # Global
    "GH", "GTK", "GNR", "GFS", # Global Health/Biotech, Sci&Tech, Natural Resources, Financial Services
    "GRE", "IRE", "GIF",       # Global / International Real Estate, Global Infrastructure
    "GX",                      # Global Flexible Portfolio (borderline: partly U.S.)
}

# Step 7 — mandate screen on the fund's (wficn) modal lipper_obj_cd.
# lipper_asset_cd == "EQ" is not a mandate screen: it admits mixed-asset, convertible,
# commodity and alternative strategies.
EXCLUDE_OBJ = [
    "B", "I", "CV", "CMD",                                                  # mixed-asset / non-equity
    "ABR", "AED", "AGM", "AMS", "EMN", "LSE", "MFF", "OS",                  # alternatives (cash is collateral, not drag)
    "GL", "GS", "GX", "GRE", "GTK", "IF", "IS", "EM", "EU", "JA", "LT", "XJ", "FM",  # non-U.S. / global
    "FX",                                                                   # flexible/tactical allocation
]

**Helpers.**

In [ ]:
def modal_code(s):
    """Most frequent non-missing value (alphabetically first on ties); None if all missing."""
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else None


ATTRITION = []   # rows of Table 1: (stage, fund-months, funds)

def log_step(label, df, id_col):
    ATTRITION.append((label, len(df), df[id_col].nunique()))
    print(f"[Table 1] {label:<45} {len(df):>11,} obs  {df[id_col].nunique():>7,} {id_col}")

## 1. Load CRSP data and merge characteristics point-in-time

Characteristics from `fund_summary2` are attached with an as-of merge: each fund-month receives the most
recent characteristics record dated on or before it. This avoids look-ahead bias from using final-period
flags (e.g. a fund reclassified as index in 2018 would otherwise lose its pre-2018 active history).

In [ ]:
raw_panel = read_sas_cached("monthly_tna_ret_nav.sas7bdat")      # caldt, crsp_fundno, mtna, mret, mnav
raw_panel["caldt"] = pd.to_datetime(raw_panel["caldt"])
log_step("Raw sample (2000–2022)", raw_panel, "crsp_fundno")

CHAR_COLS = ["crsp_fundno", "caldt", "lipper_asset_cd", "lipper_obj_cd", "index_fund_flag",
             "et_flag", "vau_fund", "sales_restrict", "mgmt_cd", "mgmt_name", "per_cash", "per_com"]
chars = read_sas_cached("fund_summary2.sas7bdat", columns=CHAR_COLS)
chars["caldt"] = pd.to_datetime(chars["caldt"])
chars = chars.dropna(subset=["caldt"]).sort_values("caldt")

panel = pd.merge_asof(
    raw_panel.sort_values("caldt"),
    chars,
    on="caldt",
    by="crsp_fundno",
    direction="backward",
)
print(f"After as-of merge: {panel.shape}")
print(f"Unmatched (no characteristics): {panel['lipper_asset_cd'].isna().sum()}")

del raw_panel, chars

## 2. Share-class screens (steps 1–6)

### Steps 1–2: equity, U.S.-only, active

- **Step 1.** Keep `lipper_asset_cd == "EQ"`; drop share classes whose modal Lipper objective is international/global.
- **Step 2.** Drop fund-months flagged as index funds (`index_fund_flag` ∈ {D, B, E}) or ETFs/ETNs (`et_flag` ∈ {F, N}).
  Also drop variable-annuity underlying funds (`vau_fund == "Y"`: insurance-wrapper cash is driven by contract-holder
  liquidity, not manager discretion, contaminating the very cash term being corrected) and sales-restricted classes
  (`sales_restrict == "Y"`: captive clientele, not publicly investable).

`index_fund_flag` is blank before mid-2003, so the month-level index screen cannot see pre-2003 index months.
The share classes *ever* flagged index/ETF are recorded here and used in step 9.

In [ ]:
# ── Step 1: equity + U.S.-only ─────────────────────────────────────────────
panel_eq = panel[panel["lipper_asset_cd"] == "EQ"].copy()
del panel
print(f"After equity filter: {panel_eq.shape}")

dom_obj    = panel_eq.groupby("crsp_fundno")["lipper_obj_cd"].agg(modal_code)
intl_funds = dom_obj[dom_obj.isin(INTERNATIONAL_OBJ)].index
n_before   = panel_eq["crsp_fundno"].nunique()
panel_eq   = panel_eq[~panel_eq["crsp_fundno"].isin(intl_funds)].copy()
print(f"US-only filter: excluded {len(intl_funds)} international/global share classes "
      f"({n_before} → {panel_eq['crsp_fundno'].nunique()})")
log_step("After US-only equity filter", panel_eq, "crsp_fundno")

# Share classes ever flagged index/ETF at any date (used in step 9)
ever_flagged = panel_eq["index_fund_flag"].isin(["D", "B", "E"]) | panel_eq["et_flag"].isin(["F", "N"])
tainted = panel_eq.loc[ever_flagged, "crsp_fundno"].unique()
first_flag_year = panel_eq.loc[ever_flagged].groupby("crsp_fundno")["caldt"].min().dt.year   # appendix diagnostic

# ── Step 2: exclude index funds and ETFs ───────────────────────────────────
panel_active = panel_eq[
    (panel_eq["index_fund_flag"].isna() | (panel_eq["index_fund_flag"] == "")) &
    (panel_eq["et_flag"].isna()         | (panel_eq["et_flag"] == ""))
].copy()
del panel_eq
print(f"After index/ETF exclusion: {panel_active.shape}")

# Data-error guard only: no fund should report TNA above $1T
assert panel_active["mtna"].max() < 1_000_000, f"Implausible TNA: {panel_active['mtna'].max():.0f}M"

# ── Step 2 (cont.): exclude variable-annuity underlying and sales-restricted classes
n_before = panel_active["crsp_fundno"].nunique()
panel_active = panel_active[
    (panel_active["vau_fund"]       != "Y") &
    (panel_active["sales_restrict"] != "Y")
].copy()
print(f"VA/sales-restricted exclusion: {n_before} -> {panel_active['crsp_fundno'].nunique()} share classes")
# NB: this Table 1 row therefore reflects index/ETF AND VA/sales-restricted exclusions.
log_step("After active management filter", panel_active, "crsp_fundno")

### Steps 3–4: minimum size and missing returns

In [ ]:
panel_aum = panel_active[panel_active["mtna"] >= MIN_TNA].copy()
log_step("After $50M AUM filter", panel_aum, "crsp_fundno")

panel_clean = panel_aum[panel_aum["mret"].notna()].copy()
print(f"After dropping missing returns: {panel_clean.shape}")
del panel_aum

### Step 5: NAV-consistency screen for implausible returns

Every fund-month with `mret > 50%` is cross-checked against the return implied by the fund's own NAV series.
A split miscoded in CRSP's return calculation produces a large `mret` while the NAV falls (~1/N for an N:1 split);
those fund-months are dropped, as are fund-months with an impossible NAV ratio (|implied return| > 500%).
NAV-consistent large returns, and cases with no prior NAV to compare against, are kept.

In [ ]:
_ps = panel_clean.sort_values(["crsp_fundno", "caldt"]).copy()
_ps["mnav_lag"]    = _ps.groupby("crsp_fundno")["mnav"].shift(1)
_ps["implied_ret"] = _ps["mnav"] / _ps["mnav_lag"] - 1
flagged = _ps[_ps["mret"] > 0.5].copy()
del _ps

def classify(row):
    ir, mr = row["implied_ret"], row["mret"]
    if pd.notna(ir) and abs(ir) > 5:
        return "drop_nav_error"        # impossible NAV ratio
    if pd.isna(ir):
        return "keep_review"           # no prior NAV: no evidence against the return
    if ir < -0.30 and mr > 0.5:
        return "drop_split_error"      # NAV fell steeply while return spiked
    if mr - ir < 0.15:
        return "keep_genuine"          # NAV agrees (slack covers reinvested distributions)
    return "keep_review"               # ambiguous: keep rather than auto-delete

flagged["disposition"] = flagged.apply(classify, axis=1)
print(flagged[["crsp_fundno", "caldt", "mret", "implied_ret", "disposition"]].to_string())
print("\n", flagged["disposition"].value_counts().to_string())

bad = flagged.loc[flagged["disposition"].isin(["drop_split_error", "drop_nav_error"]),
                  ["crsp_fundno", "caldt"]]
panel_clean = panel_clean.merge(bad.assign(_bad=True), on=["crsp_fundno", "caldt"], how="left")
n_bad = int(panel_clean["_bad"].eq(True).sum())
panel_clean = panel_clean[panel_clean["_bad"].isna()].drop(columns="_bad")
print(f"\nDropped {n_bad} NAV-inconsistent fund-months. panel_clean now {panel_clean.shape}")
# NB: this Table 1 row therefore reflects the missing-return drop AND the NAV screen.
log_step("After missing-return drop", panel_clean, "crsp_fundno")

### Step 6: at least 60 consecutive monthly returns

Each share class needs one uninterrupted run of ≥ 60 monthly returns: enough degrees of freedom for the
seven-factor model and at least one business cycle (cf. Kosowski et al. 2006, who require 60 returns in total).
A gap from a suspension or reporting lapse resets the count.

In [ ]:
panel_clean = panel_clean.sort_values(["crsp_fundno", "caldt"])
run_len     = panel_clean.groupby("crsp_fundno")["caldt"].apply(max_consecutive_run)
panel_consec = panel_clean[panel_clean["crsp_fundno"].isin(run_len[run_len >= MIN_CONSEC].index)].copy()
del panel_clean
log_step("After 60-month consecutive filter", panel_consec, "crsp_fundno")

## 3. Link share classes to funds (MFLINKS `wficn`)

`wficn` is used as the fund identifier, consistent with Fama and French (2010) and the CRSP–WRDS literature.
Two groups are dropped:

- share classes mapped to more than one `wficn` in the same month (ambiguous mapping);
- share classes with no `wficn`. These are absent from MFLINKS altogether (not present under a different
  identifier format), span the whole sample period, and account for ~3% of observations.

In [ ]:
mflinks = pd.read_csv(MFLINKS_1, sep="\t", dtype={"crsp_fundno": str, "wficn": str})
mflinks["crsp_fundno_str"] = mflinks["crsp_fundno"].astype(str).str.zfill(6)

panel_consec["crsp_fundno_str"] = panel_consec["crsp_fundno"].astype(int).astype(str).str.zfill(6)
panel_linked = panel_consec.merge(mflinks[["crsp_fundno_str", "wficn"]], on="crsp_fundno_str", how="left")
del panel_consec

# Ambiguous mapping: same share class -> several wficn in the same month
n_wficn = panel_linked.groupby(["crsp_fundno_str", "caldt"])["wficn"].nunique()
ambiguous_fundnos = panel_linked.loc[
    panel_linked.set_index(["crsp_fundno_str", "caldt"]).index.isin(n_wficn[n_wficn > 1].index),
    "crsp_fundno_str"
].unique().tolist()
panel_linked = panel_linked[~panel_linked["crsp_fundno_str"].isin(ambiguous_fundnos)].copy()
print(f"Dropped {len(ambiguous_fundnos)} share classes with ambiguous wficn mapping: {ambiguous_fundnos}")

# Unmatched share classes
unmatched = panel_linked[panel_linked["wficn"].isna()]
print(f"Unmatched: {unmatched['crsp_fundno_str'].nunique()} share classes, {len(unmatched):,} obs "
      f"({len(unmatched)/len(panel_linked):.1%} of observations), "
      f"{unmatched['caldt'].min():%Y-%m} to {unmatched['caldt'].max():%Y-%m}")
print(f"  ...of which present in MFLINKS under any identifier: "
      f"{mflinks['crsp_fundno_str'].isin(unmatched['crsp_fundno_str'].unique()).sum()}")

assert_matches("Section 3.2: share classes without wficn", unmatched["crsp_fundno_str"].nunique(), 297, 0)
assert_matches("Section 3.2: % of observations without wficn", len(unmatched) / len(panel_linked) * 100, 2.9, 1)

panel_matched = panel_linked[panel_linked["wficn"].notna()].copy()
del panel_linked, unmatched
log_step("After MFLINKS merge (drop unmatched)", panel_matched, "crsp_fundno")

## 4. Collapse share classes to fund level

Monthly returns are value-weighted across a fund's share classes by **beginning-of-month (lagged) TNA**, and TNA is
summed. End-of-month TNA already contains the month's return, so weighting by it would let month-*t* returns
influence their own weights.

- `per_cash`, `per_com`, `lipper_asset_cd`: first share class's value (the appendix reports how often classes disagree).
- `lipper_obj_cd`: the fund's modal code over its whole history (used by the step-7 mandate screen).
- Fund-months where *every* class is in its first month have no lagged TNA (spurious return of 0) and are dropped.
  This removes each fund's first month, so the 60-consecutive-month screen is re-applied at `wficn` level.

In [ ]:
panel_matched = panel_matched.sort_values(["wficn", "crsp_fundno", "caldt"])
panel_matched["mtna_lagged"]     = panel_matched.groupby("crsp_fundno")["mtna"].shift(1)
panel_matched["port_tna_lagged"] = (panel_matched.groupby(["wficn", "caldt"])["mtna_lagged"]
                                    .transform(lambda x: x.sum(min_count=1)))
panel_matched["weight"]  = panel_matched["mtna_lagged"] / panel_matched["port_tna_lagged"]
panel_matched["vw_mret"] = panel_matched["weight"] * panel_matched["mret"]

df_fundlevel = panel_matched.groupby(["wficn", "caldt"]).agg(
    mret            = ("vw_mret",         "sum"),
    mtna            = ("mtna",            "sum"),
    per_cash        = ("per_cash",        "first"),
    per_com         = ("per_com",         "first"),
    lipper_asset_cd = ("lipper_asset_cd", "first"),
).reset_index()

wficn_obj = panel_matched.groupby("wficn")["lipper_obj_cd"].agg(modal_code).rename("lipper_obj_cd")
df_fundlevel = df_fundlevel.merge(wficn_obj, on="wficn", how="left")

# Drop fund-months with no lagged TNA in any share class
no_lagged_tna = (panel_matched[panel_matched["port_tna_lagged"].isna()]
                 .groupby(["wficn", "caldt"]).size().reset_index()[["wficn", "caldt"]]
                 .assign(_drop=True))
df_fundlevel = df_fundlevel.merge(no_lagged_tna, on=["wficn", "caldt"], how="left")
print(f"First-month observations dropped (no lagged TNA): {df_fundlevel['_drop'].sum()}")
df_fundlevel = df_fundlevel[df_fundlevel["_drop"].isna()].drop(columns="_drop")

# Re-apply the 60-consecutive-month screen on the analysis unit (wficn)
run_len = df_fundlevel.groupby("wficn")["caldt"].apply(max_consecutive_run)
print(f"Funds dropped by wficn-level 60-month re-check: {(run_len < MIN_CONSEC).sum()}")
df_fundlevel = df_fundlevel[df_fundlevel["wficn"].isin(run_len[run_len >= MIN_CONSEC].index)].copy()
log_step("‖ After wficn collapse + 60-month refilter", df_fundlevel, "wficn")

# wficn of funds with any share class ever flagged index/ETF (for step 9)
tainted_wficn = panel_matched.loc[panel_matched["crsp_fundno"].isin(tainted), "wficn"].dropna().unique()

## 5. Fund-level cleaning and screens (steps 7–9)

**Cash and equity weights.** CRSP stores `per_cash` and `per_com` in percentage points (1.06 = 1.06%; Stage 1b divides
by 100). Values outside [−100, 100] are data errors and set to NaN. Negative values within the range are kept:
they reflect leverage or net short positions via derivatives and are economically meaningful.

In [ ]:
lo, hi = PCT_BOUNDS
for col in ["per_cash", "per_com"]:
    clean = col + "_clean"
    df_fundlevel[clean] = df_fundlevel[col].copy()
    df_fundlevel.loc[df_fundlevel[clean] > hi, clean] = float("nan")
    df_fundlevel.loc[df_fundlevel[clean] < lo, clean] = float("nan")
    print(f"{col}: set to NaN — above {hi}: {(df_fundlevel[col] > hi).sum()} obs, "
          f"below {lo}: {(df_fundlevel[col] < lo).sum()} obs; "
          f"{clean} missing: {df_fundlevel[clean].isna().sum()}")

assert_matches("Section 3.2: per_cash set to NaN (above 100, below −100)",
               [(df_fundlevel["per_cash"] > hi).sum(), (df_fundlevel["per_cash"] < lo).sum()], [273, 299], 0)

**Step 7 — mandate screen.** Whole-fund drop on the fund's modal `lipper_obj_cd` (see `EXCLUDE_OBJ` in Section 0).
Funds with no objective code would be kept, not silently dropped.

In [ ]:
n_missing_obj = df_fundlevel.loc[df_fundlevel["lipper_obj_cd"].isna(), "wficn"].nunique()
if n_missing_obj:
    print(f"[flag] {n_missing_obj} funds have no lipper_obj_cd — retained")

df_fundlevel = df_fundlevel[~df_fundlevel["lipper_obj_cd"].isin(EXCLUDE_OBJ)].copy()
assert not set(EXCLUDE_OBJ) & set(df_fundlevel["lipper_obj_cd"].dropna().unique())
log_step("After mandate screen (lipper_obj_cd)", df_fundlevel, "wficn")

**Step 8 — equity-share screen.** Keep a fund if the time-series median of `per_com_clean` is ≥ 80%
(whole-fund drop, so fund histories are not truncated). It runs after the mandate screen, so balanced/allocation funds
are already gone. Funds with no usable `per_com` would be kept and flagged.

In [ ]:
fund_median_com = df_fundlevel.groupby("wficn")["per_com_clean"].median()
keep_wficn = fund_median_com.index[(fund_median_com >= PER_COM_FLOOR) | fund_median_com.isna()]
print(f"Funds with no usable per_com (retained): {int(fund_median_com.isna().sum())}")

df_fundlevel = df_fundlevel[df_fundlevel["wficn"].isin(keep_wficn)].copy()
log_step("After equity-share screen (per_com ≥ 80%)", df_fundlevel, "wficn")

**Step 9 — pre-coverage index-month removal.** `index_fund_flag` has no coverage before 2003-07, so funds that CRSP
later flags as index/ETF leak their pre-2003 months into the active sample. Only those pre-coverage months are removed
for the affected funds; their post-coverage months, where a blank flag is informative, are kept.

In [ ]:
mask_cut = df_fundlevel["wficn"].isin(tainted_wficn) & (df_fundlevel["caldt"] < INDEX_FLAG_COVERAGE_START)
print(f"Pre-coverage index removal: {int(mask_cut.sum())} fund-months from "
      f"{df_fundlevel.loc[mask_cut, 'wficn'].nunique()} funds")
df_fundlevel = df_fundlevel[~mask_cut].copy()
log_step("After pre-coverage index removal", df_fundlevel, "wficn")

## 6. Save and check against the thesis

In [ ]:
save_parquet(df_fundlevel, OUT_FILE)
print(f"{df_fundlevel['wficn'].nunique():,} funds | {len(df_fundlevel):,} fund-months | "
      f"{df_fundlevel['caldt'].min():%Y-%m} to {df_fundlevel['caldt'].max():%Y-%m}")
print("Columns:", list(df_fundlevel.columns))

**Table 1 — attrition waterfall.** Above the ‖ row, *Funds* counts share classes (`crsp_fundno`); from the ‖ row on,
it counts funds (`wficn`). The final row of the thesis table (common fund set, 1,950 funds) is produced in NB07.

In [ ]:
table1 = pd.DataFrame(ATTRITION, columns=["Stage", "Fund-months", "Funds"]).set_index("Stage")
print(table1.to_string(formatters={"Fund-months": "{:,}".format, "Funds": "{:,}".format}))

# Replication check: numbers reported in Table 1 of the thesis
THESIS_TABLE1 = [
    (8_870_475, 70_657), (3_696_203, 37_236), (2_862_529, 29_678), (1_178_874, 12_568),
    (1_178_381, 12_555), (  998_011,  6_791), (  968_998,  6_489), (  503_103,  3_022),
    (  406_364,  2_368), (  373_830,  2_138), (  373_387,  2_138),
]
assert_matches("Table 1: fund-months and funds", [(o, n) for _, o, n in ATTRITION], THESIS_TABLE1, 0)

**Table 3 — fund-level summary statistics.** Each fund is collapsed to its time-series mean; the table describes the
cross-fund distribution (fund-weighted).

In [ ]:
per_fund = df_fundlevel.groupby("wficn").agg(
    tna  = ("mtna",           "mean"),
    ret  = ("mret",           "mean"),
    cash = ("per_cash_clean", "mean"),
    com  = ("per_com_clean",  "mean"),
)
per_fund["ret"] *= 100   # decimal -> percent
table3 = per_fund.describe(percentiles=[.10, .25, .50, .75, .90]).T[["mean", "std", "10%", "25%", "50%", "75%", "90%"]]
table3.index = ["TNA ($M)", "Monthly return (%)", "Cash weight (%)", "Common stock (%)"]
print(table3.round(2).to_string())

assert_matches("Table 3: TNA ($M)", table3.loc["TNA ($M)"].round(), [1755, 5525, 113, 197, 489, 1353, 3615], 0)
assert_matches("Table 3: return, cash, common stock (%)", table3.iloc[1:].values,
               [[0.62, 0.38, 0.11, 0.50, 0.69, 0.85, 1.00],
                [2.39, 2.57, 0.52, 1.14, 1.94, 3.07, 4.81],
                [93.67, 5.20, 87.51, 92.51, 95.06, 96.85, 97.97]], 2)

## Appendix — diagnostics cited in Section 3.2

These cells change nothing; they document facts the write-up relies on.

In [ ]:
# A1. per_cash coverage by year (the post-2003 coverage cliff)
miss = df_fundlevel.assign(year=df_fundlevel["caldt"].dt.year).groupby("year")["per_cash_clean"].apply(lambda x: x.isna().mean())
print("Share of fund-months with missing per_cash_clean, by year:")
print((miss * 100).round(2).to_string())
post = df_fundlevel["caldt"] >= "2003-01-01"
print(f"\nMissing share, 2003 onward: {df_fundlevel.loc[post, 'per_cash_clean'].isna().mean():.2%}")
print(f"Missing share, 2004 onward: {df_fundlevel.loc[df_fundlevel['caldt'] >= '2004-01-01', 'per_cash_clean'].isna().mean():.2%}")

In [ ]:
# A2. Why step 9 is needed: when do leaking index/ETF share classes first get flagged?
leaking = panel_active.loc[panel_active["crsp_fundno"].isin(tainted), "crsp_fundno"]
print(f"Share classes ever flagged index/ETF: {len(tainted)}; still in the active sample after step 2: "
      f"{leaking.nunique()} ({len(leaking):,} fund-months)")
print("First-flag year of those share classes:")
print(first_flag_year[first_flag_year.index.isin(leaking.unique())].value_counts().sort_index().to_string())

In [ ]:
# A3. Does per_cash differ across share classes of the same fund-month? (justifies taking the first class's value)
grp = panel_matched.groupby(["wficn", "caldt"])["per_cash"]
mx, mn = grp.max(), grp.min()
disagree = (mx - mn) > 0
print(f"Fund-months where share classes report different per_cash: {disagree.sum():,} ({disagree.mean():.3%})")
print(f"  ...of which the values straddle zero:                     {(disagree & (mx > 0) & (mn < 0)).sum():,}")
print(f"  largest spread: {(mx - mn).max():.2f} pp")